In [7]:
import numpy as np
import pandas as pd
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns

import os
os.chdir("C:/Users/arttu/OneDrive/Tiedostot/Akandi")
#1. Vuodet 2000-2019.
df_raw= pd.read_csv("data/clean_and_raw/atp_matches_2000_2019_raw.csv")

In [11]:

df_clean = df_raw.copy()
#Kopioidaan data puhdistusta varten.

#2. Carpet ottelut poistetaan 
df_clean = df_clean[df_clean["surface"].isin(["Hard", "Clay", "Grass"])]
df_clean = df_clean.reset_index(drop=True)
df_clean["surface"].value_counts()

#3. poistetaan ottelun jälkeiset featuret: score minutes, JA kaikki w_ ja l_ prefiksillä alkavat sarakkeet, koska ne kertovat ottelun lopputuloksesta.

drop_cols = (
    [c for c in df_clean.columns if c.startswith(("w_", "l_"))]
    + ["score", "minutes"])

df_clean = df_clean.drop(columns=drop_cols)
#--------------------------------
#4. lisätään vuosi-muuttuja.
df_clean["tourney_date"] = pd.to_datetime(
    df_clean["tourney_date"],
    format="%Y%m%d")

df_clean["year"] = df_clean["tourney_date"].dt.year
df_clean["year"].value_counts().sort_index()

df_clean["tourney_date"].dtype


#korjataan virheellinen pituusarvot (3.0cm)
heights = pd.concat([
    df_clean["winner_ht"],
    df_clean["loser_ht"]])

heights.min(), heights.max()

df_clean = df_clean[
    (df_clean["winner_ht"].between(140, 215)) &
    (df_clean["loser_ht"].between(140, 215))]
#-------------------------------
#8. Seed- ja entry-muuttujat sisältävät runsaasti puuttuvia arvoja ja ovat osittain päällekkäisiä ranking-muuttujien kanssa, minkä vuoksi ne poistetaan.
df_clean[[
    "winner_seed", "winner_entry",
    "loser_seed", "loser_entry"
]].isna().mean() * 100

df_clean = df_clean.drop(columns=[
    "winner_seed", "winner_entry",
    "loser_seed", "loser_entry"])

#Kätisyyskorjaukset
#katsotaan A:n (ambidextrous=molempikätisyys) määrät
pd.DataFrame({
    "winner": df_clean["winner_hand"].value_counts(),
    "loser":  df_clean["loser_hand"].value_counts()}).loc[["A", "U"]]
#A yksi havainto (loser), U:ssa 2 havaintoa (winner) ja 7 havaintoa (loser).

pd.DataFrame({
    "winner": df_clean["winner_hand"].value_counts(),
    "loser":  df_clean["loser_hand"].value_counts()}
).loc[["A", "U"]]

#Luke Jenssen on ainoa A-kätinen pelaaja datasetissä.
#katsotaan seuraavaksi Unknown merkatut pelaajat.

u_rows = df_clean[
    (df_clean["winner_hand"] == "U") |
    (df_clean["loser_hand"] == "U")
]


u_players = pd.concat([
    u_rows.loc[u_rows["winner_hand"] == "U", "winner_name"],
    u_rows.loc[u_rows["loser_hand"] == "U", "loser_name"]
]).unique()

u_players
#Guillermo Olaso: right-handed, Christopher Koderisch: ei mainita internetissä jätetään unknown, Jose Hernandez right-handed.

df_clean.loc[df_clean["winner_name"] == "Guillermo Olaso", "winner_hand"] = "R"
df_clean.loc[df_clean["loser_name"] == "Guillermo Olaso", "loser_hand"] = "R"

df_clean.loc[df_clean["winner_name"] == "Jose Hernandez", "winner_hand"] = "R"
df_clean.loc[df_clean["loser_name"] == "Jose Hernandez", "loser_hand"] = "R"

#rankingit
#13% winner_rank (ja winner_rank_points) puuttuu 41% ja loser_rank (ja loser_rank_points) puuttuu 13%
df_clean[[
    "winner_rank", "loser_rank",
    "winner_rank_points", "loser_rank_points"
]].isna().mean() * 100



winner_rank           0.132056
loser_rank            0.413776
winner_rank_points    0.132056
loser_rank_points     0.413776
dtype: float64

In [17]:
rank_by_player = pd.concat([
    df_clean[["winner_name", "winner_rank"]]
        .rename(columns={"winner_name": "player", "winner_rank": "rank"}),
    df_clean[["loser_name", "loser_rank"]]
        .rename(columns={"loser_name": "player", "loser_rank": "rank"})])

no_rank_players = (
    rank_by_player
    .groupby("player")["rank"]
    .apply(lambda x: x.notna().any())
    .loc[lambda x: x == False])

no_rank_players.index.tolist()
pd.DataFrame({"player": no_rank_players.index})
#41 pelaajaa, joilla ei ole ranking-historiaa lainkaan.


pd.crosstab(
    df_clean["winner_rank"].isna(),
    df_clean["winner_rank"].shift(1).isna())
#56 k havaintoa ranking ok. 74:ssä ranking ajanhetkellä t ok, t-1 puuttuu. 73 nyt t puuttuu, t-1 taas ok. 2 puuttuu molemmissa (t, t-1).

df_clean[
    df_clean["winner_name"].isin(no_rank_players.index) |
    df_clean["loser_name"].isin(no_rank_players.index)
][["tourney_date", "winner_name", "loser_name", "winner_rank", "loser_rank"]]


,tourney_date,winner_name,loser_name,winner_rank,loser_rank


In [33]:

#imputoidaan puuttuvat rankingit pelaajan edellisellä tunnetulla rankingillä.
df_clean = df_clean.sort_values("tourney_date")

df_clean["winner_rank"] = (
    df_clean
    .groupby("winner_name")["winner_rank"]
    .ffill())

df_clean["loser_rank"] = (
    df_clean
    .groupby("loser_name")["loser_rank"]
    .ffill())

#aikaisemmat 41 pelaajaa jakautuvat 33 (winner) ja 115 (loser) otteluhavaintoon. Jätetään ne vielä toistaiseksi sellaisenaan, kuten ylhäällä todettiin.
print(df_clean[["winner_rank", "loser_rank"]].isna().sum())

#tehdään sama rank_points-sarakkeille.
df_clean["winner_rank_points"] = (
    df_clean
    .groupby("winner_name")["winner_rank_points"]
    .ffill())

df_clean["loser_rank_points"] = (
    df_clean
    .groupby("loser_name")["loser_rank_points"]
    .ffill())

#nyt tulee enää pelaajat, joilla ole lainkaan ranking-historiaa. Jätetään nämä sellaisenaan.
df_clean[[
    "winner_rank", "loser_rank",
    "winner_rank_points", "loser_rank_points"
]].isna().sum()

winner_rank    0
loser_rank     0
dtype: int64


winner_rank           0
loser_rank            0
winner_rank_points    0
loser_rank_points     0
dtype: int64

In [34]:
#Jos jollain pelaajalla ei ole ranking:ia yhdessäkään pelissä, niin tällöin voi olettaa, että pelejä ei ole kovinkaan montaa tai pelaajan ranking taso on pieni.
#ranking-arvo = pelaajan sijoitus ATP listalla -> 2000. (teoriassa sama kuin viimeinen)
#ranking-pisteet = pelaajan ATP pisteet -> 0

#alin mahdollinen ranking
df_clean[["winner_rank", "loser_rank"]] = (
    df_clean[["winner_rank", "loser_rank"]].fillna(2000))

# alin mahdollinen ranking-pistemäärä
df_clean[["winner_rank_points", "loser_rank_points"]] = (
    df_clean[["winner_rank_points", "loser_rank_points"]].fillna(0))

df_clean[[
    "winner_rank","loser_rank",
    "winner_rank_points","loser_rank_points"]].isna().sum()

winner_rank           0
loser_rank            0
winner_rank_points    0
loser_rank_points     0
dtype: int64

In [35]:
#datasetissä ei ole nyt yhtään puuttuvia havaintoja.
print(df_clean.isna().sum())
df_clean.to_csv("data/clean_and_raw/atp_matches_2000_2019_clean.csv", index=False)

tourney_id            0
tourney_name          0
surface               0
draw_size             0
tourney_level         0
tourney_date          0
match_num             0
winner_id             0
winner_name           0
winner_hand           0
winner_ht             0
winner_ioc            0
winner_age            0
loser_id              0
loser_name            0
loser_hand            0
loser_ht              0
loser_ioc             0
loser_age             0
best_of               0
round                 0
winner_rank           0
winner_rank_points    0
loser_rank            0
loser_rank_points     0
year                  0
dtype: int64
